# 05 · Registrar alerta de qualidade

Este notebook roda no caminho **falso** do If/else, quando a validação reprova a silver. Em vez de publicar uma gold com dados ruins, registramos o problema na tabela `growdev.gold.log_alertas_qualidade`, para o time investigar.

In [ ]:
dbutils.widgets.text("catalogo", "growdev", "Catálogo")
dbutils.widgets.text("job_run_id", "manual", "Id da execução do job")

CATALOGO = dbutils.widgets.get("catalogo")
JOB_RUN_ID = dbutils.widgets.get("job_run_id")

In [ ]:
# debugValue é usado quando rodamos o notebook fora de um job
pct_nulos = dbutils.jobs.taskValues.get(
    taskKey="validar_qualidade", key="pct_nulos_loan_amount", default=-1, debugValue=-1
)
print(f"% de loan_amount nulo informado pela validação: {pct_nulos}")

In [ ]:
from pyspark.sql import functions as F

alerta = spark.createDataFrame(
    [(JOB_RUN_ID, "Silver reprovada na validação de qualidade", float(pct_nulos))],
    "job_run_id string, motivo string, pct_nulos_loan_amount double",
).withColumn("registrado_em", F.current_timestamp())

alerta.write.mode("append").saveAsTable(f"{CATALOGO}.gold.log_alertas_qualidade")
display(spark.table(f"{CATALOGO}.gold.log_alertas_qualidade").orderBy(F.desc("registrado_em")))